In [1]:
print("Hello to Feature Enginnering")

Hello to Feature Enginnering


In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
      .appName("Thesis pyspark transformations")\
      .getOrCreate()
print("here")
spark

here


26/03/29 17:27:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
FE_df=spark.read.parquet("/home/ubuntu/thesis/data/processed_data/pyspark_transformations")
FE_df.show(5)
FE_df.count()

+--------------------+----------+----------+
|                Page|      Date|page_count|
+--------------------+----------+----------+
|Motion_de_censure...|2015-11-10|        45|
|Motion_de_censure...|2015-11-11|        69|
|Motion_de_censure...|2015-11-12|        74|
|Motion_de_censure...|2015-11-13|        45|
|Motion_de_censure...|2015-11-14|        41|
+--------------------+----------+----------+
only showing top 5 rows



79783550

In [4]:
from pyspark.sql.functions import lit,avg
FE_df.withColumn("moving_avg", lit(0)).show(5)


+--------------------+----------+----------+----------+
|                Page|      Date|page_count|moving_avg|
+--------------------+----------+----------+----------+
|Motion_de_censure...|2015-11-10|        45|         0|
|Motion_de_censure...|2015-11-11|        69|         0|
|Motion_de_censure...|2015-11-12|        74|         0|
|Motion_de_censure...|2015-11-13|        45|         0|
|Motion_de_censure...|2015-11-14|        41|         0|
+--------------------+----------+----------+----------+
only showing top 5 rows



In [7]:
# with window_size=2
from pyspark.sql.window import Window
from pyspark.sql.functions import avg,stddev,round,lag
window_2= Window.partitionBy("Page").orderBy("Date").rowsBetween(-1,0)
window_lag= Window.partitionBy("Page").orderBy("Date")
w2_df=FE_df.withColumn("moving_avg",round(avg("page_count").over(window_2),2))\
    .withColumn("Moving_std",round(stddev("page_count").over(window_2),2))\
    .withColumn("lag_1", lag("page_count", 1).over(window_lag))\
    .withColumn("lag_2", lag("page_count", 2).over(window_lag))
print("data frame with window size=2")   
w2_df.show()


data frame with window size=2


[Stage 8:>                                                          (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+
|                Page|      Date|page_count|moving_avg|Moving_std|lag_1|lag_2|
+--------------------+----------+----------+----------+----------+-----+-----+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|  144| NULL|
|"""Weird_Al""_Yan...|2015-07-03|        44|      67.5|     33.23|   91|  144|
|"""Weird_Al""_Yan...|2015-07-04|        40|      42.0|      2.83|   44|   91|
|"""Weird_Al""_Yan...|2015-07-05|        29|      34.5|      7.78|   40|   44|
|"""Weird_Al""_Yan...|2015-07-06|        25|      27.0|      2.83|   29|   40|
|"""Weird_Al""_Yan...|2015-07-07|        49|      37.0|     16.97|   25|   29|
|"""Weird_Al""_Yan...|2015-07-08|        97|      73.0|     33.94|   49|   25|
|"""Weird_Al""_Yan...|2015-07-09|       239|     168.0|    100.41|   97|   49|
|"""Weird_Al""_Yan...|2015-07-10|        33|     136

In [8]:
# with window size=3
from pyspark.sql.window import Window
from pyspark.sql.functions import round
window_vis= Window.partitionBy("Page").orderBy("Date").rowsBetween(-2,0)
FE_df.withColumn("moving_avg", round(avg("page_count").over(window_vis),2))


DataFrame[Page: string, Date: date, page_count: string, moving_avg: double]

In [9]:
from pyspark.sql.window import Window
from pyspark.sql.functions import stddev
window_sd=Window.partitionBy("Page").orderBy("Date")
FE_df.withColumn("Moving_std",stddev("page_count").over(window_sd))

DataFrame[Page: string, Date: date, page_count: string, Moving_std: double]

In [10]:
# combining both moving avg and moving std 
from pyspark.sql.window import Window
from pyspark.sql.functions import stddev,round
window_vis= Window.partitionBy("Page").orderBy("Date").rowsBetween(-2,0)
w3_df=FE_df.withColumn("Moving_avg",round(avg("page_count").over(window_vis),2))\
    .withColumn("Moving_std",round(stddev("page_count").over(window_vis),2))

w3_df.show()

[Stage 11:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+
|                Page|      Date|page_count|Moving_avg|Moving_std|
+--------------------+----------+----------+----------+----------+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|
|"""Weird_Al""_Yan...|2015-07-03|        44|      93.0|     50.03|
|"""Weird_Al""_Yan...|2015-07-04|        40|     58.33|     28.36|
|"""Weird_Al""_Yan...|2015-07-05|        29|     37.67|      7.77|
|"""Weird_Al""_Yan...|2015-07-06|        25|     31.33|      7.77|
|"""Weird_Al""_Yan...|2015-07-07|        49|     34.33|     12.86|
|"""Weird_Al""_Yan...|2015-07-08|        97|      57.0|     36.66|
|"""Weird_Al""_Yan...|2015-07-09|       239|    128.33|      98.8|
|"""Weird_Al""_Yan...|2015-07-10|        33|     123.0|    105.43|
|"""Weird_Al""_Yan...|2015-07-11|        26|     99.33|    121.01|
|"""Weird_Al""_Yan...|2015-07-12|        41|     33.33|      7

In [11]:
# FE_df=FE_df.fillna({"Moving_avg":0,"Moving_std":0})
# FE_df.show()

In [12]:
# combining all features for a window size 3
from pyspark.sql.functions import stddev,round
from pyspark.sql.functions import lag
from pyspark.sql.window import Window
window_lag= Window.partitionBy("Page").orderBy("Date")
window_3= Window.partitionBy("Page").orderBy("Date").rowsBetween(-2,0)
w3_df=FE_df.withColumn("Moving_avg",round(avg("page_count").over(window_3),2))\
    .withColumn("Moving_std",round(stddev("page_count").over(window_3),2))\
    .withColumn("lag_1", lag("page_count", 1).over(window_lag))\
            .withColumn("lag_2", lag("page_count", 2).over(window_lag))
print("The data frame with window size=3")
w3_df.show(10)

The data frame with window size=3


[Stage 14:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|
+--------------------+----------+----------+----------+----------+-----+-----+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|  144| NULL|
|"""Weird_Al""_Yan...|2015-07-03|        44|      93.0|     50.03|   91|  144|
|"""Weird_Al""_Yan...|2015-07-04|        40|     58.33|     28.36|   44|   91|
|"""Weird_Al""_Yan...|2015-07-05|        29|     37.67|      7.77|   40|   44|
|"""Weird_Al""_Yan...|2015-07-06|        25|     31.33|      7.77|   29|   40|
|"""Weird_Al""_Yan...|2015-07-07|        49|     34.33|     12.86|   25|   29|
|"""Weird_Al""_Yan...|2015-07-08|        97|      57.0|     36.66|   49|   25|
|"""Weird_Al""_Yan...|2015-07-09|       239|    128.33|      98.8|   97|   49|
|"""Weird_Al""_Yan...|2015-07-10|        33|     123

In [13]:
# from pyspark.sql.functions import col
# FE_df=FE_df.withColumn("trend",col("page_count")-col("lag_1"))
# FE_df.show()

In [14]:
# for window size=5
from pyspark.sql.functions import stddev,round
from pyspark.sql.functions import lag
from pyspark.sql.window import Window
window_lag= Window.partitionBy("Page").orderBy("Date")
window_5= Window.partitionBy("Page").orderBy("Date").rowsBetween(-4,0)
w5_df=FE_df.withColumn("Moving_avg",round(avg("page_count").over(window_5),2))\
    .withColumn("Moving_std",round(stddev("page_count").over(window_5),2))\
    .withColumn("lag_1", lag("page_count", 1).over(window_lag))\
    .withColumn("lag_2", lag("page_count", 2).over(window_lag))

w5_df.show(10)

[Stage 17:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|
+--------------------+----------+----------+----------+----------+-----+-----+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|  144| NULL|
|"""Weird_Al""_Yan...|2015-07-03|        44|      93.0|     50.03|   91|  144|
|"""Weird_Al""_Yan...|2015-07-04|        40|     79.75|     48.69|   44|   91|
|"""Weird_Al""_Yan...|2015-07-05|        29|      69.6|     47.89|   40|   44|
|"""Weird_Al""_Yan...|2015-07-06|        25|      45.8|     26.43|   29|   40|
|"""Weird_Al""_Yan...|2015-07-07|        49|      37.4|     10.11|   25|   29|
|"""Weird_Al""_Yan...|2015-07-08|        97|      48.0|     28.97|   49|   25|
|"""Weird_Al""_Yan...|2015-07-09|       239|      87.8|     89.24|   97|   49|
|"""Weird_Al""_Yan...|2015-07-10|        33|      88

In [15]:
# for window size=7
from pyspark.sql.functions import stddev,round
from pyspark.sql.functions import lag
from pyspark.sql.window import Window
window_lag= Window.partitionBy("Page").orderBy("Date")
window_7= Window.partitionBy("Page").orderBy("Date").rowsBetween(-6,0)
w7_df=FE_df.withColumn("Moving_avg",round(avg("page_count").over(window_7),2))\
    .withColumn("Moving_std",round(stddev("page_count").over(window_7),2))\
    .withColumn("lag_1", lag("page_count", 1).over(window_lag))\
    .withColumn("lag_2", lag("page_count", 2).over(window_lag))

w7_df.show(10)

[Stage 20:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|
+--------------------+----------+----------+----------+----------+-----+-----+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|  144| NULL|
|"""Weird_Al""_Yan...|2015-07-03|        44|      93.0|     50.03|   91|  144|
|"""Weird_Al""_Yan...|2015-07-04|        40|     79.75|     48.69|   44|   91|
|"""Weird_Al""_Yan...|2015-07-05|        29|      69.6|     47.89|   40|   44|
|"""Weird_Al""_Yan...|2015-07-06|        25|     62.17|     46.54|   29|   40|
|"""Weird_Al""_Yan...|2015-07-07|        49|     60.29|     42.78|   25|   29|
|"""Weird_Al""_Yan...|2015-07-08|        97|     53.57|     28.88|   49|   25|
|"""Weird_Al""_Yan...|2015-07-09|       239|     74.71|     76.22|   97|   49|
|"""Weird_Al""_Yan...|2015-07-10|        33|     73.

In [16]:
from pyspark.sql.functions import avg,round
print("-----for smoothness evaluation ------")
w2_df_mstm= w2_df.select(round(avg("moving_std"),2).alias("avg_std"))
print("w2_avg_std")
w2_df_mstm.show()

w3_df_mstm=w3_df.select(round(avg("moving_std"),2).alias("avg_std"))
print("w3_avg_std")
w3_df_mstm.show()

w5_df_mstm= w5_df.select(round(avg("moving_std"),2).alias("avg_std"))
print("w5_avg_std")
w5_df_mstm.show()

w7_df_mstm= w7_df.select(round(avg("moving_std"),2).alias("avg_std"))
print("w7_avg_std")
w7_df_mstm.show()


-----for smoothness evaluation ------
w2_avg_std


+-------+
|avg_std|
+-------+
| 219.81|
+-------+

w3_avg_std


+-------+
|avg_std|
+-------+
| 277.07|
+-------+

w5_avg_std


+-------+
|avg_std|
+-------+
| 349.13|
+-------+

w7_avg_std


[Stage 41:===================================================>    (11 + 1) / 12]

+-------+
|avg_std|
+-------+
| 394.99|
+-------+



In [17]:
# so from the above avgstd's as the window size increases this avg increases making system unstable and might be a chance of hiding the anamolies 


In [18]:
# for correlation between the pagecount and moving avg
from pyspark.sql.functions import corr,round
w2_corr=w2_df.select(corr("page_count","moving_avg").alias("w2_corr"))
w2_corr.show()
w3_corr=w3_df.select(corr("page_count","moving_avg").alias("w3_corr"))
w3_corr.show()
w5_corr=w5_df.select(corr("page_count","moving_avg").alias("w5_corr"))
w5_corr.show()
w7_corr=w7_df.select(corr("page_count","moving_avg").alias("w7_corr"))
w7_corr.show()


+------------------+
|           w2_corr|
+------------------+
|0.9976033313295818|
+------------------+



+-----------------+
|          w3_corr|
+-----------------+
|0.994533552120277|
+-----------------+



+------------------+
|           w5_corr|
+------------------+
|0.9886188554274837|
+------------------+



[Stage 65:====================================================>   (16 + 1) / 17]

+------------------+
|           w7_corr|
+------------------+
|0.9835097180506631|
+------------------+



In [19]:
# windowfrmae size with 2 hasa highest corelation
#this means window frame has almost extact data

In [20]:
# visual comparision
from pyspark.sql.functions import avg
from pyspark.sql.functions import round
from pyspark.sql.window import Window
w2=Window.partitionBy("Page").orderBy("Date").rowsBetween(-1,0)
w3=Window.partitionBy("Page").orderBy("Date").rowsBetween(-2,0)
w5=Window.partitionBy("Page").orderBy("Date").rowsBetween(-4,0)
w7=Window.partitionBy("Page").orderBy("Date").rowsBetween(-6,0)


M_df= FE_df.withColumn("MA2", round(avg("page_count").over(w2),2))\
      .withColumn("MA3",round(avg("page_count").over(w3),2))\
      .withColumn("MA5",round(avg("page_count").over(w5),2))\
      .withColumn("MA7",round(avg("page_count").over(w7),2))
M_df.show()

[Stage 71:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+-----+------+-----+-----+
|                Page|      Date|page_count|  MA2|   MA3|  MA5|  MA7|
+--------------------+----------+----------+-----+------+-----+-----+
|"""Weird_Al""_Yan...|2015-07-01|       144|144.0| 144.0|144.0|144.0|
|"""Weird_Al""_Yan...|2015-07-02|        91|117.5| 117.5|117.5|117.5|
|"""Weird_Al""_Yan...|2015-07-03|        44| 67.5|  93.0| 93.0| 93.0|
|"""Weird_Al""_Yan...|2015-07-04|        40| 42.0| 58.33|79.75|79.75|
|"""Weird_Al""_Yan...|2015-07-05|        29| 34.5| 37.67| 69.6| 69.6|
|"""Weird_Al""_Yan...|2015-07-06|        25| 27.0| 31.33| 45.8|62.17|
|"""Weird_Al""_Yan...|2015-07-07|        49| 37.0| 34.33| 37.4|60.29|
|"""Weird_Al""_Yan...|2015-07-08|        97| 73.0|  57.0| 48.0|53.57|
|"""Weird_Al""_Yan...|2015-07-09|       239|168.0|128.33| 87.8|74.71|
|"""Weird_Al""_Yan...|2015-07-10|        33|136.0| 123.0| 88.6|73.14|
|"""Weird_Al""_Yan...|2015-07-11|        26| 29.5| 99.33| 88.8|71.14|
|"""Weird_Al""_Yan..

In [21]:
#unitl here i have donw until window size=3
# i need to do window size=2
# i need to window size=5
# i need to do window size=7
# train one model on those 
# getting accuracy, precesion, F1 score, Recall
# compare these window sizes
# define extra features for features engineering
# choose important ones that effects tghe system states
# end of present week

# for next week i need to divide it inot system states 
# search for models 

In [22]:
M_df.select("Page").distinct().show(5,truncate=False)

[Stage 72:==========================================================(4 + 0) / 4]

+----------------------------------------------------------------------+
|Page                                                                  |
+----------------------------------------------------------------------+
|Movimiento_(física)_es.wikipedia.org_all-access_all-agents            |
|Mr._Robot_(Fernsehserie)_de.wikipedia.org_desktop_all-agents          |
|Muireadhach_Ua_Flaithbheartaigh_en.wikipedia.org_all-access_all-agents|
|Médaille_olympique_fr.wikipedia.org_all-access_spider                 |
|N-gramme_fr.wikipedia.org_all-access_spider                           |
+----------------------------------------------------------------------+
only showing top 5 rows



In [23]:
# this is Moving average deviation 
from pyspark.sql.functions import abs,avg,col
mad_df = M_df.select(
    avg(abs(col("page_count") - col("MA2"))).alias("mad_w2"),
    avg(abs(col("page_count") - col("MA3"))).alias("mad_w3"),
    avg(abs(col("page_count") - col("MA5"))).alias("mad_w5"),
    avg(abs(col("page_count") - col("MA7"))).alias("mad_w7")
)

mad_df.show()


[Stage 77:====================================================>   (16 + 1) / 17]

+-----------------+------------------+-----------------+-----------------+
|           mad_w2|            mad_w3|           mad_w5|           mad_w7|
+-----------------+------------------+-----------------+-----------------+
|155.1491976497561|224.96319824377596|294.6824903123677|319.6283619211187|
+-----------------+------------------+-----------------+-----------------+



In [24]:
# so from the above methods 
# i have computed std deviations for smoothness evaluation( w2-219.81,w3-277.07,w5-349.13,w7-394.99)
# i have computed corrrelation between the page count and moving avgs for each window sizes (w2-0.9976,w3-0.9945,w5-0.9886,w7-0.9835)
# i hve computed moving average deviation ( original- MA)
# for w2=155.15, w3=224.96, w5-294.68,w7-319.63
# so standared deviation increases with increasing with window size resulting over smoothing with increasing the size 
# correlation decreases with the increase in the window size resulting deviations from the original data
# Mean Absolute deviation inceased from 155 to 319.63 resulting increasing with increased window size(greater distortion) 
# so window size =3 is optimal for my data which has all balanced features with SD=277,w3=0.9945,MAD=223.96

In [25]:
# final PDF

from pyspark.sql.functions import col,round
final_df = w3_df.withColumn(
    "trend", col("page_count") - col("lag_1")
)\
.withColumn(
    "deviation", round(col("page_count") - col("Moving_avg"),2)
)\
.withColumn(
    "z_score",round(col("deviation") / col("Moving_std") ,2)
)

final_df.show()

[Stage 83:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+------+---------+-------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2| trend|deviation|z_score|
+--------------------+----------+----------+----------+----------+-----+-----+------+---------+-------+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL|  NULL|      0.0|   NULL|
|"""Weird_Al""_Yan...|2015-07-02|        91|     117.5|     37.48|  144| NULL| -53.0|    -26.5|  -0.71|
|"""Weird_Al""_Yan...|2015-07-03|        44|      93.0|     50.03|   91|  144| -47.0|    -49.0|  -0.98|
|"""Weird_Al""_Yan...|2015-07-04|        40|     58.33|     28.36|   44|   91|  -4.0|   -18.33|  -0.65|
|"""Weird_Al""_Yan...|2015-07-05|        29|     37.67|      7.77|   40|   44| -11.0|    -8.67|  -1.12|
|"""Weird_Al""_Yan...|2015-07-06|        25|     31.33|      7.77|   29|   40|  -4.0|    -6.33|  -0.81|
|"""Weird_Al""_Yan...|2015-07-07|        49|     34.33|     12.8

In [26]:
final_df.printSchema()

root
 |-- Page: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- page_count: string (nullable = true)
 |-- Moving_avg: double (nullable = true)
 |-- Moving_std: double (nullable = true)
 |-- lag_1: string (nullable = true)
 |-- lag_2: string (nullable = true)
 |-- trend: double (nullable = true)
 |-- deviation: double (nullable = true)
 |-- z_score: double (nullable = true)



In [27]:
final_df.write.mode("overwrite").parquet("/home/ubuntu/thesis/data/processed_data/feature_engineering")